In [5]:
import pandas as pd
import oracledb

# =========================
# 2) 엑셀 불러오기
# =========================
file_path = "./data/공고원본_POSTING_ID포함_정리결과.xlsx"
sheet_name = "job_posting_processed"

df = pd.read_excel(file_path, sheet_name=sheet_name)

# 필요한 컬럼만 추출
update_df = df[["POSTING_ID", "TECH_STACK_NORMALIZED"]].copy()

# 컬럼명 DB 기준으로 맞추기
update_df = update_df.rename(columns={"TECH_STACK_NORMALIZED": "TECH_STACK"})

# =========================
# 3) 데이터 전처리
# =========================
# POSTING_ID 숫자형 변환
update_df["POSTING_ID"] = pd.to_numeric(update_df["POSTING_ID"], errors="coerce")

# TECH_STACK 문자열 정리
update_df["TECH_STACK"] = update_df["TECH_STACK"].fillna("").astype(str).str.strip()

# POSTING_ID 없는 행 제거
update_df = update_df[update_df["POSTING_ID"].notna()].copy()

# POSTING_ID 정수형 변환
update_df["POSTING_ID"] = update_df["POSTING_ID"].astype(int)

# 빈 문자열은 None으로 바꾸기
update_df["TECH_STACK"] = update_df["TECH_STACK"].replace("", None)

# 중복 POSTING_ID 확인
dup_ids = update_df[update_df.duplicated(subset=["POSTING_ID"], keep=False)].sort_values("POSTING_ID")
if not dup_ids.empty:
    print("중복 POSTING_ID가 있음. 아래 확인 필요:")
    print(dup_ids.to_string(index=False))
    raise ValueError("POSTING_ID 중복 때문에 안전하게 업데이트할 수 없음.")

print("업데이트 대상 행 수:", len(update_df))
print(update_df.head())

# =========================
# 4) Oracle 연결
# =========================

conn = oracledb.connect(
    user='campus_24KDT_LI8_p2_1',
    password='smhrd1',
    dsn="project-db-campus.smhrd.com:1524/xe"
)
cur = conn.cursor()

# =========================
# 5) 업데이트 SQL
# =========================
sql = """
UPDATE JOB_POSTING
SET TECH_STACK = :tech_stack
WHERE POSTING_ID = :posting_id
"""

data = [
    {
        "tech_stack": row["TECH_STACK"],
        "posting_id": row["POSTING_ID"]
    }
    for _, row in update_df.iterrows()
]

# executemany 실행
cur.executemany(sql, data)
conn.commit()

print("업데이트 완료")
print("총 반영 시도 건수:", len(data))

# =========================
# 6) 샘플 확인
# =========================
sample_ids = update_df["POSTING_ID"].head(5).tolist()

for pid in sample_ids:
    cur.execute("""
        SELECT POSTING_ID, TECH_STACK
        FROM JOB_POSTING
        WHERE POSTING_ID = :pid
    """, {"pid": pid})
    
    row = cur.fetchone()
    print(row)

cur.close()
conn.close()
print("작업 종료")

업데이트 대상 행 수: 884
   POSTING_ID                                         TECH_STACK
0           1        Android, Java, Python, MySQL, AI, 머신러닝, 딥러닝
1           2                       HTML, CSS, Java, JSP, Spring
2           3                                               클라우드
3           4  CSS, Django, Docker, Spring Boot, Spring Frame...
4           5                                        SQL, Python
업데이트 완료
총 반영 시도 건수: 884
(1, 'Android, Java, Python, MySQL, AI, 머신러닝, 딥러닝')
(2, 'HTML, CSS, Java, JSP, Spring')
(3, '클라우드')
(4, 'CSS, Django, Docker, Spring Boot, Spring Framework')
(5, 'SQL, Python')
작업 종료


In [3]:
import oracledb

oracledb.init_oracle_client(
    lib_dir=r"C:\instantclient-basic-windows.x64-23.26.1.0.0\instantclient_23_0"
)